# AI Coach — POC: Import Garmin Activities via Garmin Connect API

Credentials werden sicher per Eingabe abgefragt. Die Aktivitäten werden direkt von
Garmin Connect heruntergeladen und als `.fit` Dateien in `data/activities/` gespeichert.

In [1]:
%pip install garminconnect fitparse pandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import getpass
from pathlib import Path
import fitparse
import pandas as pd
from garminconnect import Garmin

# --- Config ---
ACTIVITIES_DIR = Path("../data/activities")
ACTIVITIES_DIR.mkdir(parents=True, exist_ok=True)
NUM_ACTIVITIES = 10  # Anzahl der zuletzt abgerufenen Aktivitäten

# --- Login ---
email = input("Garmin Connect E-Mail: ")
password = getpass.getpass("Passwort: ")

client = Garmin(email, password)
client.login()
print("Login erfolgreich.")

# --- FIT-Dateien herunterladen ---
activities = client.get_activities(0, NUM_ACTIVITIES)

for act in activities:
    act_id = act["activityId"]
    act_name = act.get("activityName", str(act_id))
    fit_path = ACTIVITIES_DIR / f"{act_id}.fit"
    if not fit_path.exists():
        fit_data = client.download_activity(act_id, dl_fmt=client.ActivityDownloadFormat.ORIGINAL)
        fit_path.write_bytes(fit_data)
        print(f"  Heruntergeladen: {fit_path.name}  ({act_name})")
    else:
        print(f"  Bereits vorhanden: {fit_path.name}")

fit_files = sorted(ACTIVITIES_DIR.glob("*.fit"))
print(f"\nGesamt: {len(fit_files)} .fit Datei(en) in {ACTIVITIES_DIR.resolve()}")

mobile+cffi returned 429: Mobile login returned 429 — IP rate limited by Garmin
mobile+requests failed: HTTPSConnectionPool(host='sso.garmin.com', port=443): Max retries exceeded with url: /mobile/api/login?clientId=GCM_IOS_DARK&locale=en-US&service=https%3A%2F%2Fmobile.integration.garmin.com%2Fgcm%2Fios (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1081)')))


Login erfolgreich.
  Heruntergeladen: 22715678237.fit  (Viernheim Laufen)
  Heruntergeladen: 22691212409.fit  (Viernheim Radfahren)
  Heruntergeladen: 22675360493.fit  (Indoor Cycling)
  Heruntergeladen: 22651476092.fit  (Viernheim Radfahren)
  Heruntergeladen: 22631636721.fit  (Indoor Cycling)
  Heruntergeladen: 22607649768.fit  (Viernheim Laufen)
  Heruntergeladen: 22572922875.fit  (Viernheim Radfahren)
  Heruntergeladen: 22563039853.fit  (Viernheim - Tempo)
  Heruntergeladen: 22550078368.fit  (Weinheim Wandern)
  Heruntergeladen: 22524864188.fit  (Viernheim Laufen)

Gesamt: 10 .fit Datei(en) in G:\Andere Computer\Mein Computer\GoogleDrive\privat\Code_Repo\AI_Coach\data\activities


In [ ]:
def parse_fit_file(fit_path: Path) -> pd.DataFrame:
    """Parse a single .fit file and return a DataFrame of 'record' messages."""
    fit = fitparse.FitFile(str(fit_path))
    rows = []
    for msg in fit.get_messages("record"):
        row = {data.name: data.value for data in msg}
        rows.append(row)
    return pd.DataFrame(rows)


# Parse the first available file as a quick test
if fit_files:
    df = parse_fit_file(fit_files[0])
    print(f"Parsed: {fit_files[0].name}  →  {len(df)} records, {len(df.columns)} fields")
    print("\nColumns:", list(df.columns))
    df.head()
else:
    print("No .fit files found. Copy files from your Garmin device to:", ACTIVITIES_DIR.resolve())

In [ ]:
# Parse ALL .fit files into a single combined DataFrame
all_dfs = []
for f in fit_files:
    df_tmp = parse_fit_file(f)
    df_tmp.insert(0, "source_file", f.name)
    all_dfs.append(df_tmp)

if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)
    print(f"Total records across all activities: {len(df_all)}")
    df_all.head()
else:
    print("No .fit files to process.")